# QBUS6810 M4 — Boosting and time variable comparison

Use the shared chronological Train and Validation CSVs. Select each Boosting family directly by validation WAE 2:1; repeat selection for WAE 1:1 and 1:2 as sensitivity analyses. Test data is not read. M2 supplies the official linear results and M3 supplies RF/DT results; this notebook does not produce competing official numbers for those families.

Run from the repository root. Required packages: numpy, pandas, scipy, scikit-learn, lightgbm 4.6.0 and optuna 4.5.0. The searches each use 24 trials and seed 6810. Outputs go to data/result/boosting/ and data/result/time_trend_comparison/inputs/.

In [1]:
import hashlib, json, platform, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 6810
N_THREADS = 2
N_LGB_TRIALS = 24
MAX_BOOST_ROUNDS = 1400
EARLY_STOPPING_ROUNDS = 100
SPLIT_DIR = Path('data/split')
BOOST_OUT = Path('data/result/boosting')
TIME_OUT = Path('data/result/time_trend_comparison/inputs')
BOOST_OUT.mkdir(parents=True, exist_ok=True)
TIME_OUT.mkdir(parents=True, exist_ok=True)
SCENARIOS = {
    'Main - Overvaluation Priority (2:1)': ('WAE_2_1', 2., 1.),
    'Alternative - Equal Priority (1:1)': ('WAE_1_1', 1., 1.),
    'Alternative - Undervaluation Priority (1:2)': ('WAE_1_2', 1., 2.),
}
PRIMARY_METRIC = 'WAE_2_1'
manifest = json.loads((SPLIT_DIR / 'manifest.json').read_text(encoding='utf-8'))
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
print('Python', platform.python_version(), 'sklearn', sklearn.__version__, 'LightGBM', lgb.__version__, 'Optuna', optuna.__version__)

Python 3.12.14 sklearn 1.7.2 LightGBM 4.6.0 Optuna 4.5.0


## Inputs and transaction alignment

The official CSV hashes come from the current manifest. Membership files supply transaction IDs and dates in the same row order. The repaired month indicators are checked against those dates; no month values are reconstructed from price or row number.

In [3]:
frames = {}
members = {}
for split, expected_rows in [('train', 54853), ('validation', 18132)]:
    csv_path = SPLIT_DIR / manifest['splits'][split]['file']
    member_path = SPLIT_DIR / manifest['membership'][split]['file']
    assert sha256_file(csv_path) == manifest['splits'][split]['sha256']
    assert sha256_file(member_path) == manifest['membership'][split]['sha256']
    frame = pd.read_csv(csv_path)
    member = pd.read_csv(member_path, dtype={'transaction_id':'string', 'property_id':'string'})
    member['sale_date'] = pd.to_datetime(member['sale_date'], errors='raise')
    assert len(frame) == len(member) == expected_rows
    assert frame.shape[1] == 211 and frame.columns.is_unique
    assert frame.select_dtypes(exclude='number').empty
    assert np.isfinite(frame.to_numpy(dtype=float)).all()
    assert frame['sale_price'].gt(0).all()
    np.testing.assert_allclose(frame['log_sale_price'], np.log1p(frame['sale_price']), rtol=1e-12)
    assert member['transaction_id'].is_unique
    month_cols = [f'month_{month}' for month in range(2, 13)]
    expected_month = member['sale_date'].dt.month.to_numpy()
    actual_month = np.where(frame[month_cols].sum(axis=1).to_numpy() == 0, 1,
                            np.arange(2, 13)[frame[month_cols].to_numpy().argmax(axis=1)])
    assert frame[month_cols].isin([0, 1]).all().all()
    assert frame[month_cols].sum(axis=1).le(1).all()
    assert np.array_equal(actual_month, expected_month)
    np.testing.assert_allclose(frame['house_age'] + frame['char_yrblt'],
                               member['sale_date'].dt.year.to_numpy(), rtol=0, atol=1e-8)
    frames[split], members[split] = frame, member
assert frames['train'].columns.tolist() == frames['validation'].columns.tolist()
FEATURES = [column for column in frames['train'] if column not in {'sale_price', 'log_sale_price'}]
assert len(FEATURES) == 209 and not {'transaction_id', 'property_id'} & set(FEATURES)
X_train_A = frames['train'][FEATURES].copy()
X_valid_A = frames['validation'][FEATURES].copy()
y_train = frames['train']['sale_price'].to_numpy(dtype=float)
y_valid = frames['validation']['sale_price'].to_numpy(dtype=float)
validation_ids = members['validation']['transaction_id'].astype(str).to_numpy()
validation_dates = members['validation']['sale_date']
print('Train', X_train_A.shape, 'Validation', X_valid_A.shape, 'membership and month encoding checked')

Train (54853, 209) Validation (18132, 209) membership and month encoding checked


## Dollar scale metrics and WAE search

Positive prediction minus actual price is overvaluation. All three WAE values divide by the number of transactions. Predictions and early stopping are evaluated after inverse transformation for log target trials.

In [5]:
def nonnegative_prediction(values):
    values = np.asarray(values, dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('Prediction contains NaN or infinity')
    return np.maximum(values, 0.)

def dollars(values, target):
    return nonnegative_prediction(np.expm1(values) if target == 'log1p' else values)

def metrics(actual, prediction):
    actual = np.asarray(actual, dtype=float)
    prediction = nonnegative_prediction(prediction)
    error = prediction - actual
    result = {
        'MSE': float(mean_squared_error(actual, prediction)),
        'RMSE': float(np.sqrt(mean_squared_error(actual, prediction))),
        'R2': float(r2_score(actual, prediction)),
        'MAE': float(mean_absolute_error(actual, prediction)),
        'mean_signed_error': float(error.mean()),
        'median_ratio': float(np.median(prediction / actual)),
        'share_overvalued': float(np.mean(error > 0)),
    }
    for _, (key, over_weight, under_weight) in SCENARIOS.items():
        result[key] = float(np.mean(np.abs(error) * np.where(error > 0, over_weight, under_weight)))
    return result

assert np.isclose(metrics([100, 100, 100], [120, 70, 100])['WAE_2_1'], 70 / 3)

def lgb_model(params, rounds=MAX_BOOST_ROUNDS):
    return lgb.LGBMRegressor(objective='regression', metric='None', n_estimators=rounds,
        random_state=SEED, n_jobs=N_THREADS, verbosity=-1, deterministic=True,
        force_col_wise=True, **params)

def search_lightgbm(X_train, X_valid, variant):
    records, predictions, histories = [], {}, {}
    def objective(trial):
        target = trial.suggest_categorical('target', ['raw', 'log1p'])
        depth = trial.suggest_categorical('max_depth', [4, 6, 8, -1])
        leaves = trial.suggest_int('num_leaves', 8, 48)
        if depth > 0:
            leaves = min(leaves, 2 ** depth)
        params = dict(max_depth=depth, num_leaves=leaves,
            learning_rate=trial.suggest_float('learning_rate', .025, .15, log=True),
            min_child_samples=trial.suggest_int('min_child_samples', 30, 180),
            subsample=trial.suggest_float('subsample', .7, 1.), subsample_freq=1,
            colsample_bytree=trial.suggest_float('colsample_bytree', .65, 1.),
            reg_alpha=trial.suggest_float('reg_alpha', 1e-5, 10., log=True),
            reg_lambda=trial.suggest_float('reg_lambda', .01, 100., log=True))
        train_target = np.log1p(y_train) if target == 'log1p' else y_train
        valid_target = np.log1p(y_valid) if target == 'log1p' else y_valid
        model = lgb_model(params)
        history = {}
        def dollar_wae_2_1(eval_target, eval_prediction):
            actual = np.expm1(eval_target) if target == 'log1p' else eval_target
            prediction = dollars(eval_prediction, target)
            error = prediction - actual
            return 'dollar_WAE_2_1', float(np.mean(np.abs(error) * np.where(error > 0, 2., 1.))), False
        start = time.time()
        model.fit(X_train, train_target,
            eval_set=[(X_valid, valid_target)], eval_names=['validation'],
            eval_metric=dollar_wae_2_1,
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, first_metric_only=True, verbose=False),
                       lgb.record_evaluation(history)])
        rounds = int(model.best_iteration_ or MAX_BOOST_ROUNDS)
        prediction = dollars(model.predict(X_valid, num_iteration=rounds), target)
        candidate_id = ('C' if variant == 'A_no_time' else 'T') + f'{trial.number + 1:03d}'
        row = dict(candidate_id=candidate_id, model_family='LightGBM', variant=variant,
            feature_set=variant, target=target, configuration=f'{candidate_id} ({target}, {rounds} trees)',
            params=json.dumps(params, sort_keys=True), n_estimators=rounds,
            n_predictors=X_train.shape[1], elapsed_seconds=time.time() - start,
            **metrics(y_valid, prediction))
        records.append(row)
        predictions[candidate_id] = prediction
        histories[candidate_id] = history
        trial.set_user_attr('candidate_id', candidate_id)
        trial.set_user_attr('best_iteration', rounds)
        print(variant, candidate_id, 'WAE 2:1', f'{row[PRIMARY_METRIC]:,.2f}', 'trees', rounds, flush=True)
        return row[PRIMARY_METRIC]
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=8))
    base = dict(max_depth=6, num_leaves=31, learning_rate=.05, min_child_samples=60,
                subsample=.9, colsample_bytree=.9, reg_alpha=.01, reg_lambda=1.)
    study.enqueue_trial(dict(target='raw', **base))
    study.enqueue_trial(dict(target='log1p', **base))
    study.optimize(objective, n_trials=N_LGB_TRIALS, n_jobs=1, show_progress_bar=False)
    return pd.DataFrame(records), predictions, histories, study

optuna.logging.set_verbosity(optuna.logging.WARNING)
A_trials, A_predictions, A_histories, A_study = search_lightgbm(X_train_A, X_valid_A, 'A_no_time')
A_trials.to_csv(BOOST_OUT / 'lightgbm_A_validation_trials.csv', index=False)
A_study.trials_dataframe().to_csv(BOOST_OUT / 'lightgbm_A_optuna_trials.csv', index=False)
A_winner = A_trials.sort_values([PRIMARY_METRIC, 'candidate_id']).iloc[0]
print('A LightGBM WAE 2:1 winner:', A_winner['configuration'])

A_no_time C001 WAE 2:1 133,245.98 trees 742
A_no_time C002 WAE 2:1 127,840.51 trees 820
A_no_time C003 WAE 2:1 134,350.22 trees 1391
A_no_time C004 WAE 2:1 128,238.49 trees 1145
A_no_time C005 WAE 2:1 127,700.48 trees 831
A_no_time C006 WAE 2:1 133,746.44 trees 554
A_no_time C007 WAE 2:1 127,131.07 trees 787
A_no_time C008 WAE 2:1 127,931.21 trees 844
A_no_time C009 WAE 2:1 128,489.37 trees 253
A_no_time C010 WAE 2:1 127,571.59 trees 1398
A_no_time C011 WAE 2:1 134,089.93 trees 1397
A_no_time C012 WAE 2:1 127,584.89 trees 1332
A_no_time C013 WAE 2:1 127,911.52 trees 1400
A_no_time C014 WAE 2:1 127,614.30 trees 650
A_no_time C015 WAE 2:1 128,026.46 trees 791
A_no_time C016 WAE 2:1 127,791.34 trees 905
A_no_time C017 WAE 2:1 127,702.55 trees 1147
A_no_time C018 WAE 2:1 127,642.10 trees 1317
A_no_time C019 WAE 2:1 134,778.60 trees 890
A_no_time C020 WAE 2:1 128,278.25 trees 327
A_no_time C021 WAE 2:1 127,567.06 trees 402
A_no_time C022 WAE 2:1 127,821.74 trees 426
A_no_time C023 WAE 2:1 1

## Boosting family comparison

Histogram Boosting retains the six prespecified reference configurations from the earlier M4 notebook. It is selected by WAE within its own family. RF/DT and linear models are excluded here because M3 and M2 own their official results.

In [7]:
hist_records, hist_predictions = [], {}
hist_grid = [(200,.05,15), (400,.05,31), (400,.1,15)]
for target in ['raw', 'log1p']:
    for rounds, rate, leaves in hist_grid:
        params = dict(max_iter=rounds, learning_rate=rate, max_leaf_nodes=leaves,
                      min_samples_leaf=50, l2_regularization=10.)
        model = HistGradientBoostingRegressor(random_state=SEED, early_stopping=False, **params)
        train_target = np.log1p(y_train) if target == 'log1p' else y_train
        start = time.time()
        model.fit(X_train_A, train_target)
        prediction = dollars(model.predict(X_valid_A), target)
        candidate_id = f'H{len(hist_records) + 1:03d}'
        row = dict(candidate_id=candidate_id, model_family='HistGradientBoosting', variant='A_no_time',
            feature_set='A_no_time', target=target,
            configuration=f'{candidate_id} ({target}, {rounds} trees)',
            params=json.dumps(params, sort_keys=True), n_estimators=rounds,
            n_predictors=len(FEATURES), elapsed_seconds=time.time() - start,
            **metrics(y_valid, prediction))
        hist_records.append(row)
        hist_predictions[candidate_id] = prediction
        print('HistBoost', candidate_id, 'WAE 2:1', f'{row[PRIMARY_METRIC]:,.2f}', flush=True)
Hist_trials = pd.DataFrame(hist_records)
Hist_trials.to_csv(BOOST_OUT / 'histboost_validation_candidates.csv', index=False)
all_candidates = pd.concat([A_trials, Hist_trials], ignore_index=True)
all_candidates.to_csv(BOOST_OUT / 'M4_boosting_candidates.csv', index=False)

HistBoost H001 WAE 2:1 141,423.03
HistBoost H002 WAE 2:1 135,424.22
HistBoost H003 WAE 2:1 136,010.28
HistBoost H004 WAE 2:1 131,508.31
HistBoost H005 WAE 2:1 129,111.29
HistBoost H006 WAE 2:1 129,214.08


## Scenario specific selection and fixed prediction comparison

Each family selects its own best configuration separately for the three validation WAE scenarios. The main scenario then selects one Boosting model for the fixed prediction comparison with the training mean baseline. The MSE champion table is supplementary.

In [9]:
scenario_rows = []
for family, group in all_candidates.groupby('model_family', sort=True):
    for scenario, (metric, _, _) in SCENARIOS.items():
        winner = group.sort_values([metric, 'candidate_id']).iloc[0]
        scenario_rows.append(dict(model_family=family, scenario=scenario,
            winning_configuration=winner['configuration'], feature_set='A_no_time',
            selected_WAE=winner[metric], MSE=winner['MSE'], R2=winner['R2'], MAE=winner['MAE']))
scenario_columns = ['model_family','scenario','winning_configuration','feature_set','selected_WAE','MSE','R2','MAE']
scenario_winners = pd.DataFrame(scenario_rows)[scenario_columns]
scenario_winners.to_csv(BOOST_OUT / 'scenario_validation_winners.csv', index=False)
mse_champions = all_candidates.sort_values(['MSE','candidate_id']).drop_duplicates('model_family')
mse_champions.to_csv(BOOST_OUT / 'MSE_champions_supplement.csv', index=False)
primary_winner = all_candidates.sort_values([PRIMARY_METRIC,'candidate_id']).iloc[0]
primary_id = primary_winner['candidate_id']
primary_prediction = (A_predictions if primary_winner['model_family'] == 'LightGBM' else hist_predictions)[primary_id]
mean_prediction = np.full(len(y_valid), y_train.mean())
fixed_4c = []
for scenario, (metric, _, _) in SCENARIOS.items():
    model_score = metrics(y_valid, primary_prediction)[metric]
    baseline_score = metrics(y_valid, mean_prediction)[metric]
    fixed_4c.append(dict(scenario=scenario, model_family=primary_winner['model_family'],
        winning_configuration=primary_winner['configuration'], selected_WAE=model_score,
        baseline_WAE=baseline_score, reduction_pct=100 * (1 - model_score / baseline_score)))
pd.DataFrame(fixed_4c).to_csv(BOOST_OUT / '4c_fixed_prediction_comparison.csv', index=False)
selected_ids = set()
for _, family_candidates in all_candidates.groupby('model_family'):
    for metric, _, _ in SCENARIOS.values():
        selected_ids.add(family_candidates.sort_values([metric, 'candidate_id']).iloc[0]['candidate_id'])
selected_ids = sorted(selected_ids)
prediction_export = pd.DataFrame({'transaction_id':validation_ids, 'actual_sale_price':y_valid})
for candidate_id in selected_ids:
    prediction_export[candidate_id] = (A_predictions if candidate_id.startswith('C') else hist_predictions)[candidate_id]
prediction_export.to_csv(BOOST_OUT / 'M4_selected_validation_predictions.csv', index=False)
print(scenario_winners.to_string(index=False))
print('4c primary model:', primary_winner['model_family'], primary_winner['configuration'])

        model_family                                    scenario   winning_configuration feature_set  selected_WAE          MSE       R2           MAE
HistGradientBoosting         Main - Overvaluation Priority (2:1) H005 (log1p, 400 trees)   A_no_time 129111.293119 4.792241e+10 0.748742 102426.052340
HistGradientBoosting          Alternative - Equal Priority (1:1)   H002 (raw, 400 trees)   A_no_time  99834.425334 4.502701e+10 0.763923  99834.425334
HistGradientBoosting Alternative - Undervaluation Priority (1:2)   H002 (raw, 400 trees)   A_no_time 164079.059836 4.502701e+10 0.763923  99834.425334
            LightGBM         Main - Overvaluation Priority (2:1) C007 (log1p, 787 trees)   A_no_time 127131.073862 4.793727e+10 0.748664 101136.852115
            LightGBM          Alternative - Equal Priority (1:1)   C001 (raw, 742 trees)   A_no_time 100015.434961 4.520069e+10 0.763012 100015.434961
            LightGBM Alternative - Undervaluation Priority (1:2)  C003 (raw, 1391 trees)   A_n

## Time variable: fixed hyperparameters

A uses the 209 shared predictors. B appends months_since_2023_01 from membership sale_date. B uses the A LightGBM WAE 2:1 winner's exact parameters and number of trees. Both fixed fits disable early stopping; this isolates the month variable.

In [11]:
def month_index(dates):
    return ((dates.dt.year - 2023) * 12 + dates.dt.month - 1).astype(int).to_numpy()
train_month_index = month_index(members['train']['sale_date'])
valid_month_index = month_index(validation_dates)
assert train_month_index.min() == 0 and train_month_index.max() == 17
assert valid_month_index.min() == 18 and valid_month_index.max() == 23
X_train_B = X_train_A.assign(months_since_2023_01=train_month_index)
X_valid_B = X_valid_A.assign(months_since_2023_01=valid_month_index)
assert X_train_A.shape[1] == 209 and X_train_B.shape[1] == 210
fixed_params = json.loads(A_winner['params'])
fixed_target = A_winner['target']
fixed_rounds = int(A_winner['n_estimators'])
fixed_predictions = {}
for variant, train_features, valid_features in [
    ('A_no_time', X_train_A, X_valid_A), ('B_month_index', X_train_B, X_valid_B)]:
    model = lgb_model(fixed_params, rounds=fixed_rounds)
    model.fit(train_features, np.log1p(y_train) if fixed_target == 'log1p' else y_train)
    fixed_predictions[variant] = dollars(model.predict(valid_features, num_iteration=fixed_rounds), fixed_target)
    print('Fixed', variant, 'WAE 2:1', f'{metrics(y_valid, fixed_predictions[variant])[PRIMARY_METRIC]:,.2f}')

Fixed A_no_time WAE 2:1 127,131.07
Fixed B_month_index WAE 2:1 135,179.81


## Time variable: equal budget retuning

B receives the same 24 Optuna trials, seed, WAE 2:1 objective and early stopping rule as A. A's already completed WAE search supplies its retuned row; no second A search is needed.

In [13]:
B_trials, B_predictions, B_histories, B_study = search_lightgbm(X_train_B, X_valid_B, 'B_month_index')
B_trials.to_csv(BOOST_OUT / 'lightgbm_B_validation_trials.csv', index=False)
B_study.trials_dataframe().to_csv(BOOST_OUT / 'lightgbm_B_optuna_trials.csv', index=False)
B_winner = B_trials.sort_values([PRIMARY_METRIC,'candidate_id']).iloc[0]
print('B LightGBM WAE 2:1 winner:', B_winner['configuration'])

B_month_index T001 WAE 2:1 149,934.23 trees 59
B_month_index T002 WAE 2:1 137,584.32 trees 77
B_month_index T003 WAE 2:1 153,224.44 trees 109
B_month_index T004 WAE 2:1 138,771.67 trees 151
B_month_index T005 WAE 2:1 135,783.62 trees 71
B_month_index T006 WAE 2:1 145,964.67 trees 738
B_month_index T007 WAE 2:1 136,596.97 trees 134
B_month_index T008 WAE 2:1 135,443.43 trees 732
B_month_index T009 WAE 2:1 137,666.77 trees 252
B_month_index T010 WAE 2:1 137,222.31 trees 646
B_month_index T011 WAE 2:1 148,247.31 trees 297
B_month_index T012 WAE 2:1 135,582.85 trees 66
B_month_index T013 WAE 2:1 137,216.91 trees 347
B_month_index T014 WAE 2:1 136,009.48 trees 91
B_month_index T015 WAE 2:1 136,222.38 trees 453
B_month_index T016 WAE 2:1 136,254.13 trees 59
B_month_index T017 WAE 2:1 138,304.06 trees 404
B_month_index T018 WAE 2:1 135,927.23 trees 1397
B_month_index T019 WAE 2:1 146,379.32 trees 502
B_month_index T020 WAE 2:1 136,871.36 trees 96
B_month_index T021 WAE 2:1 136,627.52 trees 65

## Group comparison export and predeclared decision

The fixed comparison is the primary isolation check. Adopt B only if its validation WAE 2:1 improves by more than 1%; otherwise retain A. If MAE improves while WAE worsens, retain A and report that tradeoff. Trees cannot extrapolate beyond the observed training month indices; B can at most distinguish the validation period from earlier training months. A already contains year information through house_age and char_yrblt.

In [15]:
comparison_rows = []
comparison_predictions = {}
for comparison, variant, prediction, configuration, count in [
    ('fixed_hyperparameters','A_no_time',fixed_predictions['A_no_time'], A_winner['configuration'], 209),
    ('fixed_hyperparameters','B_month_index',fixed_predictions['B_month_index'], A_winner['configuration'], 210),
    ('retuned','A_no_time',A_predictions[A_winner['candidate_id']], A_winner['configuration'], 209),
    ('retuned','B_month_index',B_predictions[B_winner['candidate_id']], B_winner['configuration'], 210),
]:
    comparison_predictions[(comparison, variant)] = prediction
    comparison_rows.append(dict(model_family='LightGBM', variant=variant, comparison=comparison,
        configuration=configuration, n_predictors=count, **metrics(y_valid, prediction)))
comparison_columns = ['model_family','variant','comparison','configuration','n_predictors',
    'MSE','RMSE','R2','MAE','WAE_2_1','WAE_1_1','WAE_1_2',
    'mean_signed_error','median_ratio','share_overvalued']
time_comparison = pd.DataFrame(comparison_rows)[comparison_columns]
time_comparison.to_csv(TIME_OUT / 'M4_boosting_time_trend.csv', index=False)
fixed_A = time_comparison.query("comparison == 'fixed_hyperparameters' and variant == 'A_no_time'").iloc[0]
fixed_B = time_comparison.query("comparison == 'fixed_hyperparameters' and variant == 'B_month_index'").iloc[0]
improvement_pct = 100 * (fixed_A['WAE_2_1'] - fixed_B['WAE_2_1']) / fixed_A['WAE_2_1']
decision = 'adopt_B' if improvement_pct > 1 else 'retain_A'
tradeoff = bool(fixed_B['MAE'] < fixed_A['MAE'] and fixed_B['WAE_2_1'] > fixed_A['WAE_2_1'])
monthly_rows = []
for (comparison, variant), prediction in comparison_predictions.items():
    for month in range(7, 13):
        mask = validation_dates.dt.month.to_numpy() == month
        monthly_rows.append(dict(model_family='LightGBM', variant=variant, comparison=comparison,
            validation_month=f'2024-{month:02d}', n=int(mask.sum()),
            mean_signed_error=float(np.mean(prediction[mask] - y_valid[mask]))))
pd.DataFrame(monthly_rows).to_csv(BOOST_OUT / 'M4_monthly_signed_error.csv', index=False)
time_predictions = pd.DataFrame({'transaction_id':validation_ids,'actual_sale_price':y_valid})
for (comparison, variant), prediction in comparison_predictions.items():
    time_predictions[f'{comparison}_{variant}'] = prediction
time_predictions.to_csv(BOOST_OUT / 'M4_time_validation_predictions.csv', index=False)
summary = dict(selection_metric='2024H2 validation WAE_2_1', A_winner=A_winner['configuration'],
    B_winner=B_winner['configuration'], fixed_improvement_pct=improvement_pct,
    decision=decision, MAE_improves_while_WAE_worsens=tradeoff,
    train_month_range=[0,17], validation_month_range=[18,23], test_read=False)
(BOOST_OUT / 'M4_decision.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
print(time_comparison.to_string(index=False))
print('Fixed comparison WAE improvement:',f'{improvement_pct:.3f}%','decision:',decision,'MAE/WAE tradeoff:',tradeoff)

model_family       variant            comparison           configuration  n_predictors          MSE          RMSE       R2           MAE       WAE_2_1       WAE_1_1       WAE_1_2  mean_signed_error  median_ratio  share_overvalued
    LightGBM     A_no_time fixed_hyperparameters C007 (log1p, 787 trees)           209 4.793727e+10 218945.804966 0.748664 101136.852115 127131.073862 101136.852115 176279.482483      -49148.408621      0.903771          0.335484
    LightGBM B_month_index fixed_hyperparameters C007 (log1p, 787 trees)           210 4.543352e+10 213151.400869 0.761791  95886.619813 135179.812749  95886.619813 152480.046689      -17300.233939      0.979140          0.460898
    LightGBM     A_no_time               retuned C007 (log1p, 787 trees)           209 4.793727e+10 218945.804966 0.748664 101136.852115 127131.073862 101136.852115 176279.482483      -49148.408621      0.903771          0.335484
    LightGBM B_month_index               retuned T008 (log1p, 732 trees)        